# BLS Preprocessing

In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from scipy.spatial.distance import pdist
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [2]:
warnings.filterwarnings("ignore")

RAW_DIR = Path("../../data/raw/bls")
OUT_DIR = Path("../../data/processed/bls")
FINAL_FILE = OUT_DIR / "bls_processed.csv"

FILES = {
    2019: RAW_DIR / "oesm19nat.xlsx",
    2022: RAW_DIR / "oesm22nat.xlsx",
    2024: RAW_DIR / "oesm24nat.xlsx",
}


## Aggregation

### 1. Load

In [3]:
frames = []
for year, path in FILES.items():
    df = pd.read_excel(path, sheet_name=0)
    df.columns = df.columns.str.upper().str.strip()
    df["YEAR"] = year
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)


In [4]:
detail = raw[raw["O_GROUP"] == "detailed"].copy()

WAGE_COLS = [
    "H_MEAN", "A_MEAN",
    "H_PCT10", "H_PCT25", "H_MEDIAN", "H_PCT75", "H_PCT90",
    "A_PCT10", "A_PCT25", "A_MEDIAN", "A_PCT75", "A_PCT90",
]

for col in WAGE_COLS:
    detail[col] = pd.to_numeric(detail[col], errors="coerce")
detail["TOT_EMP"] = pd.to_numeric(detail["TOT_EMP"], errors="coerce")

detail["MAJOR_SOC"] = detail["OCC_CODE"].str[:2]
major_titles = (
    raw[raw["O_GROUP"] == "major"]
    .drop_duplicates("OCC_CODE")
    .set_index("OCC_CODE")["OCC_TITLE"]
    .to_dict()
)
detail["MAJOR_TITLE"] = detail["MAJOR_SOC"].map(
    lambda code: major_titles.get(f"{code}-0000", "Unknown")
)

detail.head()


,AREA,AREA_TITLE,AREA_TYPE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,O_GROUP,...,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY,YEAR,PRIM_STATE,PCT_RPT,MAJOR_SOC,MAJOR_TITLE
4,99,U.S.,1,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,detailed,...,184460.0,NaN,NaN,NaN,NaN,2019,NaN,NaN,11,Management Occupations
6,99,U.S.,1,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,detailed,...,100780.0,157430.0,NaN,NaN,NaN,2019,NaN,NaN,11,Management Occupations
8,99,U.S.,1,0,Cross-industry,cross-industry,1235,11-1031,Legislators,detailed,...,29270.0,75520.0,100470.0,True,NaN,2019,NaN,NaN,11,Management Occupations
11,99,U.S.,1,0,Cross-industry,cross-industry,1235,11-2011,Advertising and Promotions Managers,detailed,...,125510.0,175940.0,NaN,NaN,NaN,2019,NaN,NaN,11,Management Occupations
13,99,U.S.,1,0,Cross-industry,cross-industry,1235,11-2021,Marketing Managers,detailed,...,136850.0,185320.0,NaN,NaN,NaN,2019,NaN,NaN,11,Management Occupations


### 2. Statistical Aggregation

In [5]:
def safe_mode(s):
    m = s.dropna().mode()
    return m.iloc[0] if len(m) else np.nan

stat_agg = (
    detail.groupby(["MAJOR_SOC", "MAJOR_TITLE", "YEAR"])
    .agg(
        emp_sum=("TOT_EMP", "sum"),
        wage_mean=("A_MEAN", "mean"),
        wage_median=("A_MEAN", "median"),
        wage_mode=("A_MEAN", safe_mode),
        wage_std=("A_MEAN", "std"),
        wage_var=("A_MEAN", "var"),
        wage_q1=("A_MEAN", lambda x: np.nanpercentile(x, 25)),
        wage_q3=("A_MEAN", lambda x: np.nanpercentile(x, 75)),
    )
    .reset_index()
    .round(0)
)

stat_agg.head()

,MAJOR_SOC,MAJOR_TITLE,YEAR,emp_sum,wage_mean,wage_median,wage_mode,wage_std,wage_var,wage_q1,wage_q3
0,11,Management Occupations,2019,8054130,110622.0,113755.0,49440.0,33885.0,1.148185e+09,83072.0,133815.0
1,11,Management Occupations,2022,9860710,118032.0,115410.0,57610.0,39372.0,1.550135e+09,84335.0,145098.0
2,11,Management Occupations,2024,10966830,126781.0,125240.0,62640.0,42248.0,1.784904e+09,94630.0,154830.0
3,13,Business and Financial Operations Occupations,2019,8183760,74340.0,71570.0,49550.0,15099.0,2.279769e+08,65640.0,80220.0
4,13,Business and Financial Operations Occupations,2022,9677710,83041.0,80840.0,51650.0,19416.0,3.769677e+08,72705.0,88805.0


### 3. Temporal Aggregation

In [6]:
temporal_agg = (
    stat_agg.groupby(["MAJOR_SOC", "MAJOR_TITLE"])
    .agg(
        emp_mean=("emp_sum", "mean"),
        wage_mean=("wage_mean", "mean"),
        wage_std=("wage_mean", "std"),
    )
    .reset_index()
    .round(0)
)

temporal_agg.head()

,MAJOR_SOC,MAJOR_TITLE,emp_mean,wage_mean,wage_std
0,11,Management Occupations,9627223.0,118478.0,8089.0
1,13,Business and Financial Operations Occupations,9404300.0,82671.0,8152.0
2,15,Computer and Mathematical Occupations,4916557.0,106098.0,8520.0
3,17,Architecture and Engineering Occupations,2547017.0,91254.0,6554.0
4,19,"Life, Physical, and Social Science Occupations",1350017.0,86447.0,5664.0


### 4. Save

In [7]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Aggregation complete. Final processed file will be saved at the end.")

Aggregation complete. Final processed file will be saved at the end.


## Sampling

### 1. Random Sample (10% Rule)

In [8]:
N = len(detail)
n = int(N * 0.10)

sample = detail.sample(n=n, random_state=42)

print(f"Population: {N}  |  Sample: {n}  |  Ratio: {n/N:.0%}")
sample.head()

Population: 2450  |  Sample: 245  |  Ratio: 10%


,AREA,AREA_TITLE,AREA_TYPE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,O_GROUP,...,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY,YEAR,PRIM_STATE,PCT_RPT,MAJOR_SOC,MAJOR_TITLE
2175,99,U.S.,1,0,Cross-industry,cross-industry,1235,43-3011,Bill and Account Collectors,detailed,...,39470.0,48290.0,59610.0,NaN,NaN,2022,US,NaN,43,Office and Administrative Support Occupations
1674,99,U.S.,1,0,Cross-industry,cross-industry,1235,23-2011,Paralegals and Legal Assistants,detailed,...,59200.0,75560.0,94960.0,NaN,NaN,2022,US,NaN,23,Legal Occupations
3992,99,U.S.,1,0,Cross-industry,cross-industry,1235,51-8031,Water and Wastewater Treatment Plant and Syste...,detailed,...,58260.0,71280.0,86160.0,NaN,NaN,2024,US,NaN,51,Production Occupations
1089,99,U.S.,1,0,Cross-industry,cross-industry,1235,51-2041,Structural Metal Fabricators and Fitters,detailed,...,40390.0,50130.0,61500.0,NaN,NaN,2019,NaN,NaN,51,Production Occupations
1035,99,U.S.,1,0,Cross-industry,cross-industry,1235,49-3052,Motorcycle Mechanics,detailed,...,37600.0,48530.0,60060.0,NaN,NaN,2019,NaN,NaN,49,"Installation, Maintenance, and Repair Occupations"


### 2. CLT
Distribution of Sample Means

In [9]:
wages = detail["A_MEAN"].dropna()

n_samples = 1000
sample_means = [wages.sample(n=n, replace=False).mean() for _ in range(n_samples)]
sample_means = pd.Series(sample_means)

pop_mu = wages.mean()
mu_xbar = sample_means.mean()
se = wages.std() / np.sqrt(n)

print(f"Population μ:            {pop_mu:,.0f}")
print(f"Mean of sample means μ_x̄: {mu_xbar:,.0f}")
print(f"Std of sample means:     {sample_means.std():,.0f}")
print(f"σ/√n:                    {se:,.0f}")

Population μ:            71,020
Mean of sample means μ_x̄: 71,081
Std of sample means:     2,725
σ/√n:                    2,889


### 3. Save

In [10]:
print("Sampling complete. Results will be included in the final processed file.")

Sampling complete. Results will be included in the final processed file.


## Discretization

### 1. Equal-Width Binning

In [11]:
detail["wage_eqw"] = pd.cut(
    detail["A_MEAN"], bins=5,
    labels=["Very Low", "Low", "Mid", "High", "Very High"],
)

detail["wage_eqw"].value_counts().sort_index()

wage_eqw
Very Low     2145
Low           231
Mid            37
High           18
Very High       5
Name: count, dtype: int64

### 2. Equal-Frequency Binning

In [12]:
detail["wage_eqf"] = pd.qcut(
    detail["A_MEAN"], q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
)

detail["wage_eqf"].value_counts().sort_index()

wage_eqf
Q1    488
Q2    487
Q3    488
Q4    486
Q5    487
Name: count, dtype: int64

### 3. Entropy-Based Discretization

In [13]:
mask = detail["A_MEAN"].notna()
X = detail.loc[mask, ["A_MEAN"]]
y = LabelEncoder().fit_transform(detail.loc[mask, "MAJOR_SOC"])

tree = DecisionTreeClassifier(criterion="entropy", max_leaf_nodes=5, random_state=42)
tree.fit(X, y)

cuts = sorted(set(tree.tree_.threshold[tree.tree_.threshold != -2]))
detail["wage_entropy"] = pd.cut(detail["A_MEAN"], bins=[-np.inf] + cuts + [np.inf])

detail["wage_entropy"].value_counts().sort_index()

wage_entropy
(-inf, 40715.0]        454
(40715.0, 60460.0]     816
(60460.0, 87500.0]     622
(87500.0, 171905.0]    471
(171905.0, inf]         73
Name: count, dtype: int64

### 4. Clustering-Based

In [14]:
X_km = detail["A_MEAN"].dropna().values.reshape(-1, 1)
km = KMeans(n_clusters=5, random_state=42, n_init=10).fit(X_km)

centers = sorted(km.cluster_centers_.flatten())
midpoints = [(centers[i] + centers[i + 1]) / 2 for i in range(len(centers) - 1)]

detail["wage_cluster"] = pd.cut(detail["A_MEAN"], bins=[-np.inf] + midpoints + [np.inf])

detail["wage_cluster"].value_counts().sort_index()

wage_cluster
(-inf, 61812.893]           1311
(61812.893, 101799.419]      776
(101799.419, 179850.373]     283
(179850.373, 294651.12]       46
(294651.12, inf]              20
Name: count, dtype: int64

### 5. Save

In [15]:
print("Discretization complete. New columns will be exported in the final processed file.")

Discretization complete. New columns will be exported in the final processed file.


## Data Reduction

### 1. PCA (SVD)

In [16]:
wage_df = detail[WAGE_COLS].dropna()
X_scaled = StandardScaler().fit_transform(wage_df)

pca = PCA().fit(X_scaled)
cumvar = np.cumsum(pca.explained_variance_ratio_)

n95 = np.argmax(cumvar >= 0.95) + 1
print(f"Original features: {X_scaled.shape[1]}")
print(f"Components for 95% variance: {n95}")
print(f"Variance per component: {pca.explained_variance_ratio_.round(3)}")

Original features: 12
Components for 95% variance: 1
Variance per component: [0.952 0.04  0.005 0.002 0.001 0.001 0.    0.    0.    0.    0.    0.   ]


In [17]:
pca_reduced = PCA(n_components=n95).fit_transform(X_scaled)

reduced_df = pd.DataFrame(
    pca_reduced,
    columns=[f"PC{i+1}" for i in range(n95)],
    index=wage_df.index,
)

print(f"Reduced shape: {reduced_df.shape}")
reduced_df.head()

Reduced shape: (2155, 1)


,PC1
18,5.557934
25,6.980314
27,8.771097
29,5.303097
33,8.648987


### 2. Save

In [18]:
final_df = detail.copy()

# Keep sample membership from the 10% sampling step.
final_df["is_random_sample_10pct"] = final_df.index.isin(sample.index).astype(int)

# Add PCA components where wage values were available.
final_df = final_df.join(reduced_df, how="left")

final_df.to_csv(FINAL_FILE, index=False)
print(f"Saved single processed file: {FINAL_FILE}")
print(f"Rows: {len(final_df):,} | Columns: {final_df.shape[1]}")

Saved single processed file: ../../data/processed/bls/bls_processed.csv
Rows: 2,450 | Columns: 41
